In [ ]:
from datasets import load_dataset

# Load the Hugging Face documentation dataset
dataset = load_dataset("m-ric/huggingface_doc", split="train")

print(f"Number of documents: {len(dataset)}")

In [ ]:
!pip install -q gradio langchain langchain-classic langchain-core langchain-community langchain-text-splitters langchain-groq sentence-transformers faiss-cpu datasets


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys
import html
import re
import gradio as gr

# LangChain Modular Packages
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_groq import ChatGroq
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_classic.retrievers import ContextualCompressionRetriever
from datasets import load_dataset


# Secure Colab Key Fetching

from google.colab import userdata
try:
    groq_api_key = userdata.get('GROQ_API_KEY')
    os.environ["GROQ_API_KEY"] = groq_api_key
except Exception:
    groq_api_key = ""

try:
    hf_token = userdata.get('HF_TOKEN')
    os.environ["HF_TOKEN"] = hf_token
except Exception:
    hf_token = ""

# Assert credentials exist before proceeding

if not groq_api_key:
    raise ValueError("❌ `GROQ_API_KEY` not found in Colab Secrets! Add it via the sidebar key icon.")

# Directory where your index will be persisted locally

INDEX_SAVE_DIR = "/content/drive/MyDrive/faiss_hf_docs_index"

# Initializing global embedding spaces and loading model
embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [5]:
# Data ingestion and vector store creation

def data_ingestion(embeddings_model):

    """
    Checks if a persisted FAISS index exists on disk.
    Loads it if found, otherwise generates it from scratch and caches it.
    """

    if os.path.exists(INDEX_SAVE_DIR):

        print(f"📦 [INDEX CACHE FOUND] Loading existing FAISS vector index from: './{INDEX_SAVE_DIR}'...")

        # allow_dangerous_deserialization is required for loading local pickle/binary FAISS indices safely
        vector_store = FAISS.load_local(INDEX_SAVE_DIR, embeddings_model, allow_dangerous_deserialization=True)

        print("🎉 FAISS Index loaded successfully. Skipped text generation pipeline steps!")

        return vector_store

    print("✨ [NO INDEX CACHE FOUND] Constructing Vector Store from scratch...")

    try:

        # Download dataset from HuggingFace. Loading 200 documents to speed up processing
        dataset = load_dataset("m-ric/huggingface_doc", split="train[:200]")
        raw_documents = []
        for index, row in enumerate(dataset):
            text = row["text"].strip()
            if len(text) > 100:
                raw_documents.append(Document(page_content=text, metadata={"source": f"hf_doc_chunk_{index}"}))

        print(f"Successfully collected {len(raw_documents)} raw documentation instances.")

        text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        chunks = text_splitter.split_documents(raw_documents)

        print(f"Split raw docs into {len(chunks)} segments.")

        # Compiling Dense Embeddings and Building FAISS Vector Index
        vector_store = FAISS.from_documents(chunks, embeddings_model)

        # Save index locally so next runs can skip this entire block
        vector_store.save_local(INDEX_SAVE_DIR)

        print("FAISS Vector Store caching complete.")

        return vector_store

    except Exception as e:
        print(f"[ERROR IN INGESTION] Failure encountered during data mapping initialization: {e}")
        return None

In [6]:
# Instantiate Two-Stage RAG Pipeline

def initialize_retriever(vector_store):
    """
    Creates a retriever which can retrieve documents from vector store using fast bi-encoder (uses HNSW) and
    re-rank them using cross-encoder.
    """

    base_retriever = vector_store.as_retriever(search_kwargs={"k": 12})

    # Instantiating Stage-2 Cross-Encoder Reranker Model
    cross_encoder_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
    reranker = CrossEncoderReranker(model=cross_encoder_model, top_n=3)

    # Compiling LangChain Contextual Compression Pipeline
    retriever = ContextualCompressionRetriever(
        base_compressor=reranker, base_retriever=base_retriever
    )

    print(f"Two-stage RAG pipeline instantiated successfully.")

    return retriever

In [7]:
# Query (user query + context) execution via LLM

def process_query_llm(user_query, retrieved_docs):
    """
    Instantiates LLM client, processes query and formats retrieved documents.
    """

    if not user_query.strip():
        return "Please input a valid search statement query.", ""

    llm_client = ChatGroq(temperature=0.0, groq_api_key=groq_api_key, model_name="openai/gpt-oss-20b")
    print("🎉 Engine components ready for requests.\n")
    print(f"\n[INFERENCE PIPELINE] Processing Request: '{user_query}'")

    prompt_template = ChatPromptTemplate.from_template("""
    You are an authoritative AI Assistant specialized in technical framework documentation.
    Answer the user's inquiry concisely and strictly according to the provided context references.

    Context References:
    {context}

    Question: {question}

    Answer:""")

    context_str = "\n\n".join([doc.page_content for doc in retrieved_docs])
    rag_chain = prompt_template | llm_client | StrOutputParser()

    response = rag_chain.invoke({"context": context_str, "question": user_query})

    print("[INFERENCE PIPELINE] RAG Chain finished generation process successfully.\n")

    # Format output to display in context panels
    context_doc = ""
    for idx, doc in enumerate(retrieved_docs):
        context_doc += f"### Chunk Rank #{idx+1} (Source: {doc.metadata.get('source')}):\n{doc.page_content}\n\n---\n\n"

    return response, context_doc

In [ ]:
def complete_rag_pipeline(query):
    """
    Executes entire pipeline as per below sequential manner:
    1. Ingests data and creates vector store
    2. Intialize retriever. Filters top_n=3 candidates for context creation.
    3. Invokes retriever.
    4. Executes final query (user query + context) using LLM call.

    Args:
        query: user query to RAG system

    Return: answer, retrieved_docs
    """

    if not query.strip():
        return "Please enter a question.", ""

    try:

        vector_store = data_ingestion(embeddings_model)

        if vector_store is not None:

            retriever_pipeline = initialize_retriever(vector_store)

            # Retrieving Context Documents via Bi-Encoder and filtering with Cross-Encoder
            retrieved_results = retriever_pipeline.invoke(query)

            print(f"Filtered down to {len(retrieved_results)} hyper-relevant context structures.")

            print("Establishing Communication Channel with Groq Cloud Endpoint...")

            # Generate answer
            answer, retrieved_docs = process_query_llm(query, retrieved_results)

            return answer, retrieved_docs

        else:
          print("❌ Fatal pipeline component creation setup abort configuration error.")

    except Exception as e:
        error_msg = f"Error: {str(e)}"
        return error_msg, error_msg


# Gradio interface for RAG system.
with gr.Blocks(title="Advanced RAG System", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🤗 Retrieval-Augmented Generation (RAG) System

    Ask questions about the Hugging Face documentation and get grounded, accurate answers!

    This system uses:
    - **Bi-Encoder**: Fast HNSW-based vector search
    - **Cross-Encoder**: Precise reranking of candidates
    - **GPT-OSS-20B**: Grounded generation using retrieved context
    """)

    with gr.Row():
        with gr.Column(scale=2):
            query_input = gr.Textbox(
                label="Search Bar / Enter Query",
                placeholder="Enter your request regarding Hugging Face framework components...",
                lines=2
            )
            submit_btn = gr.Button("Execute Query", variant="primary")
            output_answer = gr.Markdown("## 💬 Generated Answer")

        with gr.Column(scale=1):
            output_context = gr.Markdown("## 📚 Retrieved Ranked Context Chunks Used")

    submit_btn.click(
        fn=complete_rag_pipeline,
        inputs=[query_input],
        outputs=[output_answer, output_context]
    )

    # Examples
    gr.Markdown("### 💡 Try these example questions:")
    gr.Examples(
        examples=[
            ["How do I load a pretrained model?"],
            ["What is tokenization?"],
            ["How do I fine-tune a transformer model?"],
            ["What are the different types of attention mechanisms?"],
        ],
        inputs=[query_input],
    )

    gr.Markdown("""
    ---
    ### 🎯 How it works:
    1. **Fast Retrieval**: Bi-encoder searches millions of documents using HNSW
    2. **Precise Reranking**: Cross-encoder deeply analyzes top candidates
    3. **Grounded Generation**: GPT generates answers using ONLY the retrieved documents
    """)

# Launch Application and Create Public Shareable URL Link
demo.launch(inline=True, share=True)